# Kickbase percentile ratings by minutes group

Ranks individual player appearances from the latest derived Kickbase points JSON (`generated_at`). All teams, positions, and competitions in that export are pooled. Groups are 10 ≤ minutes < 25, 25 ≤ minutes < 50, 50 ≤ minutes < 75, and 75+ minutes. Appearances below 10 minutes or with invalid points/minutes are excluded; source player eligibility is preserved.

Equal points receive average ranks. Percentile = 100 × (rank − minimum rank) / (maximum rank − minimum rank). Rating = 3.5 + 6.5 × percentile / 100, rounded half-up to one decimal. These are relative ratings from estimated Kickbase points, not actual SofaScore ratings.

A singleton or all-tied group receives percentile 50 and rating 6.8, with a prominent warning and a persisted fallback flag. Empty groups have no ratings. Player averages use team-match recency weights 28/24/20/16/12, newest first, matching the score notebook. Missing or excluded appearances are omitted and remaining weights renormalized without shifting slots. Averages are rounded half-up to one decimal; individual match percentile ratings are unchanged. Rerun this notebook to refresh previously saved unweighted averages.


In [1]:
import sys
from pathlib import Path
from IPython.display import display
import pandas as pd

def locate_project_root():
    for start in (Path.cwd().resolve(), Path(globals().get('__vsc_ipynb_file__', '.')).resolve()):
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Run from within the project directory.')
PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import importlib
import sofascore_kickbase_percentile_ratings as percentile_ratings
importlib.reload(percentile_ratings)


<module 'sofascore_kickbase_percentile_ratings' from 'C:\\kickbase project\\sofascore_kickbase_percentile_ratings.py'>

In [2]:
result = percentile_ratings.export_ratings()
print('Source:', result['document']['points_source_file'])
print('Source generated:', result['document']['points_generated_at'])
print('JSON:', result['json_path'])
print('CSV:', result['csv_path'])
display(pd.Series(result['document']['exclusions'], name='Appearances').to_frame())
display(pd.Series(result['document']['group_counts'], name='Appearances').to_frame())
if result['document']['fallback_flags']:
    print('WARNING: Fallback ratings were assigned. Review these groups:')
    display(pd.DataFrame(result['document']['fallback_flags']))
else:
    print('No fallback rating flags.')


Source: overall_player_kickbase_point_averages_2026-10-08_15-15-35_109329+0200.json
Source generated: 2026-10-08T15:15:35+02:00
JSON: C:\kickbase project\outputs\derived\sofascore_kickbase_percentile_ratings\sofascore_kickbase_percentile_ratings_2026-10-08_15-15-43_201414+0200.json
CSV: C:\kickbase project\outputs\derived\sofascore_kickbase_percentile_ratings\sofascore_kickbase_percentile_ratings_2026-10-08_15-15-43_201414+0200.csv


,Appearances
source,1042
invalid_points_or_minutes,0
below_10_minutes,44
included,998


,Appearances
10–25 min,117
25–50 min,121
50–75 min,160
75+ min,600


No fallback rating flags.


## Per-appearance ratings

In [3]:
display(result['appearances'].sort_values(['group', 'points']).reset_index(drop=True))


,team_id,team,player_id,player_name,position,match_id,minutes_played,points,group,percentile,rating,opponent
0,2527,Borussia M'gladbach,876858,Robin Hack,MID,16434028,21,-26,10–25 min,0.000000,3.5,SV 07 Elversberg
1,2598,SV 07 Elversberg,1414916,Francis Onyeka,MID,16434041,11,-25,10–25 min,0.862069,3.6,FC Bayern München
2,2569,TSG Hoffenheim,67647,Andrej Kramarić,FWD,16434019,17,-24,10–25 min,1.724138,3.6,Borussia Dortmund
3,36360,RB Leipzig,319857,Benjamin Henrichs,DEF,16434033,10,-22,10–25 min,2.586207,3.7,Bayer 04 Leverkusen
4,2527,Borussia M'gladbach,798175,Isac Lidberg,FWD,16434040,23,-19,10–25 min,3.448276,3.7,SC Freiburg
...,...,...,...,...,...,...,...,...,...,...,...,...
993,2672,FC Bayern München,108579,Harry Kane,FWD,16434038,77,355,75+ min,99.332220,10.0,1. FC Union Berlin
994,2538,SC Freiburg,1020422,Yuito Suzuki,MID,16434029,80,392,75+ min,99.499165,10.0,SV Werder Bremen
995,2538,SC Freiburg,990209,Yannick Engelhardt,MID,16434040,90,407,75+ min,99.666110,10.0,Borussia M'gladbach
996,2672,FC Bayern München,978838,Michael Olise,MID,16938858,90,525,75+ min,99.833055,10.0,Bodø/Glimt


## Player averages
One-decimal unweighted averages, retaining players with at least one valid appearance.

In [4]:
display(result['players'].sort_values('average_rating', ascending=False).reset_index(drop=True))


,team_id,team,category,player_id,player_name,position,rating_count,average_rating
0,2672,FC Bayern München,overall,978838,Michael Olise,MID,5,9.6
1,2672,FC Bayern München,overall,108579,Harry Kane,FWD,5,9.5
2,2672,FC Bayern München,overall,1142251,Aleksandar Pavlović,MID,5,9.4
3,2672,FC Bayern München,overall,259117,Joshua Kimmich,MID,5,9.2
4,2672,FC Bayern München,overall,843665,Alphonso Davies,DEF,5,9.1
...,...,...,...,...,...,...,...,...
268,2547,1. FC Union Berlin,overall,850891,Michel Aebischer,MID,4,4.0
269,2676,Hamburger SV,overall,1158529,Martin Adeline,MID,2,4.0
270,2547,1. FC Union Berlin,overall,842718,Matheo Raab,GK,1,3.9
271,2598,SV 07 Elversberg,overall,817978,Amara Condé,MID,1,3.9
